# Experiment: minGPT Adder Evaluation

This notebook loads a trained minGPT adder checkpoint and runs it on example queries.

Expected artifact:
- `/workspace/minGPT/out/adder/model.pt` on the remote host.

This notebook also works from the local mount if present:
- `/Users/jacski/gpt2-training-codex/remote_workspace/minGPT/out/adder/model.pt`


In [ ]:
from __future__ import annotations

import json
import random
import sys
from pathlib import Path

import torch

random.seed(7)
torch.manual_seed(7)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(7)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


In [ ]:
# Resolve repository and checkpoint paths.
repo_candidates = [
    Path("/workspace/minGPT"),
    Path("/Users/jacski/gpt2-training-codex/remote_workspace/minGPT"),
    Path.cwd(),
]

REPO_ROOT = next((p for p in repo_candidates if (p / "mingpt").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find minGPT repo root. Set REPO_ROOT manually.")

CHECKPOINT_PATH = REPO_ROOT / "out" / "adder" / "model.pt"
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT, CHECKPOINT_PATH


In [ ]:
from mingpt.model import GPT

# Infer ndigit from the saved config when available; fall back to 2.
config_path = REPO_ROOT / "out" / "adder" / "config.json"
if config_path.exists():
    config_dict = json.loads(config_path.read_text())
    NDIGIT = int(config_dict.get("data", {}).get("ndigit", 2))
else:
    NDIGIT = 2

model_config = GPT.get_default_config()
model_config.model_type = "gpt-nano"
model_config.vocab_size = 10
model_config.block_size = 3 * NDIGIT

model = GPT(model_config)
state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

NDIGIT


In [ ]:
def predict_sum(a: int, b: int, ndigit: int = NDIGIT) -> int:
    if not (0 <= a < 10**ndigit and 0 <= b < 10**ndigit):
        raise ValueError(f"a and b must be in [0, {10**ndigit - 1}] for ndigit={ndigit}")

    astr = f"{a:0{ndigit}d}"
    bstr = f"{b:0{ndigit}d}"
    prompt_digits = [int(ch) for ch in (astr + bstr)]
    x = torch.tensor(prompt_digits, dtype=torch.long, device=DEVICE).unsqueeze(0)

    with torch.no_grad():
        full = model.generate(x, ndigit + 1, do_sample=False)

    out_digits_reversed = full[0, - (ndigit + 1):].tolist()
    out_digits = list(reversed(out_digits_reversed))
    return int("".join(str(d) for d in out_digits))


In [ ]:
# A few concrete example queries.
examples = [(1, 2), (13, 29), (50, 50), (74, 15), (99, 1)]
rows = []
for a, b in examples:
    pred = predict_sum(a, b)
    gt = a + b
    rows.append({"a": a, "b": b, "prediction": pred, "ground_truth": gt, "correct": pred == gt})

rows


In [ ]:
# Optional: quick random-sample accuracy check.
N = 50
num_correct = 0
for _ in range(N):
    a = random.randint(0, 10**NDIGIT - 1)
    b = random.randint(0, 10**NDIGIT - 1)
    num_correct += int(predict_sum(a, b) == (a + b))

accuracy = num_correct / N
{"num_examples": N, "num_correct": num_correct, "accuracy": accuracy}
